In [171]:
### import essential libraries
from dotenv import load_dotenv

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display 
import os


###  for clean outputs, I configured seaborn
sns.set_theme(style="whitegrid")

### to see all columns and rows
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.set_option('display.width',None)
pd.set_option('display.max_colwidth',None)

# loading de dataset
df=pd.read_csv('maplefreight_delivery_delay_dataset.csv')

### Exploratoration
Before applying any transformations, I will inspect the raw structure of the dataset. I need to verify the number of rows and columns, check data types, detect missing values, and analyze the distribution of this target variable 'delivered_late', because it talks about conditions of failing.

In [172]:
# taking a look
df.info()

### these features: driver_experience_years, weather_condition, traffic_index, fuel_cost_cad shows different number of rows, probably NaN or null values
print('\n\n==> Missing values in all features:',df.isnull().sum())
print('\n\n==> Total Missing values:',df.isnull().sum().sum())
#display(df[df.isnull().any(axis=1)].head())



<class 'pandas.DataFrame'>
RangeIndex: 6035 entries, 0 to 6034
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   shipment_id                6035 non-null   str    
 1   origin_city                6035 non-null   str    
 2   destination_city           6035 non-null   str    
 3   carrier                    6035 non-null   str    
 4   service_level              6035 non-null   str    
 5   goods_type                 6035 non-null   str    
 6   customer_priority_tier     6035 non-null   str    
 7   shipment_month             6035 non-null   int64  
 8   distance_km                6035 non-null   float64
 9   weight_kg                  6035 non-null   float64
 10  num_stops                  6035 non-null   int64  
 11  is_cross_border            6035 non-null   int64  
 12  scheduled_transit_hours    6035 non-null   float64
 13  pickup_delay_minutes       6035 non-null   float64
 14  dri

In [173]:
### calculate absolute counts and percentages of missing values per column and
### display it in a table ordered by missing_data_count
### ==> from Gemini assistant <==
missing_summary = pd.DataFrame({
    'missing_data_count': df.isnull().sum(),
    'missing_data_perc': ((df.isnull().sum()/len(df))*100).round(2)
})

### filter and sort columns that contain missing values
missing_summary = missing_summary[missing_summary['missing_data_count'] > 0].sort_values(by='missing_data_count', ascending=False)

print("==> Missing values in descensing order:")
display(missing_summary)

==> Missing values in descensing order:


,missing_data_count,missing_data_perc
fuel_cost_cad,240,3.98
driver_experience_years,210,3.48
weather_condition,150,2.49
traffic_index,121,2.00


In [174]:
### checking for duplicate shipment_id entries and total duplicated rows
### as we did in previous class activities
dup_tot=df.duplicated().sum()
print('\n\n==> Duplicated rows:',dup_tot)
dup_rows=df[df['shipment_id'].duplicated(keep=False)]
#display(dup_rows.head())





==> Duplicated rows: 32


In [175]:
### exploring relationship between actual_transit_hours and delivered_late
### because actual_transit_hours is recorded after delivery happens so using it predicts the past
print("==> Displaying statistics for actual_transit_hours by delivery_status:")
### when I change .sum() to .describe()  it shows statistics
display(df.groupby('delivered_late')['actual_transit_hours'].describe())

# calculating hours percentage
late_perc= (df['delivered_late'].value_counts(normalize=True, dropna=False)*100).round(2)

print("\n==> Actual transit hours percentage:")
print(late_perc)


==> Displaying statistics for actual_transit_hours by delivery_status:


,count,mean,std,min,25%,50%,75%,max
delivered_late,,,,,,,,
0,4793.0,9.689572,4.400783,0.88,6.56,9.08,12.1400,34.24
1,1242.0,16.961047,7.465666,1.17,11.72,15.84,20.8375,55.38



==> Actual transit hours percentage:
delivered_late
0    79.42
1    20.58
Name: proportion, dtype: float64


### Insights
- Data are imbalanced, there are much more 0 than 1 values

In [176]:
### in the previous table looks like the more actual_transit_hours the more possibility to deliver late

### let's check if there is a correlation between transit hours and delay using their features names with a correlation matrix
### ==> I Googled this <==
corr_cols=[col for col in ['actual_transit_hours','scheduled_transit_hours', 'delivered_late'] if col in df.columns]
correlation = df[corr_cols].corr()

print("\n==> Correlation matrix:")
display(correlation)


==> Correlation matrix:


,actual_transit_hours,scheduled_transit_hours,delivered_late
actual_transit_hours,1.000000,0.926491,0.493526
scheduled_transit_hours,0.926491,1.000000,0.208301
delivered_late,0.493526,0.208301,1.000000


### Insights
- First of all, there is a high correlation between **actual_transit_hours** and **delivered_late** (0.49), and medium low correlation between **scheduled_transit_hours** and **delivered_late** (0.20 ish) but...
- The analysis shows that **actual_transit_hours** is recorded only after shipment delivery. In a real production environment, dispatchers need predictions <u>before</u> or <u>during</u> shipment. Therefore, including **actual_transit_hours** in our feature set would cause severe data leakage. We must drop this column before training

In [177]:
"""
According to .info() these are the categorical features

 1   origin_city                6035 non-null   str    
 2   destination_city           6035 non-null   str    
 3   carrier                    6035 non-null   str    
 4   service_level              6035 non-null   str    
 5   goods_type                 6035 non-null   str    
 6   customer_priority_tier     6035 non-null   str    
 16  weather_condition          5885 non-null   str 
"""

### check unique values in categorical features using exact dataset column names
categorical_cols = ['origin_city','destination_city','carrier','service_level','goods_type','customer_priority_tier', 'weather_condition']
### ==> I Googled this loop <==
for col in categorical_cols:
    if col in df.columns:
        print(f"\n==> Unique values in {col}:")
        print(list(df[col].unique()[:10]))




==> Unique values in origin_city:
['Mississauga ON', 'Montreal QC', 'Toronto ON', 'London ON', 'Calgary AB', 'Winnipeg MB', 'Ottawa ON']

==> Unique values in destination_city:
['Edmonton AB', 'Winnipeg MB', 'Hamilton ON', 'Sudbury ON', 'Toronto ON', 'Quebec City QC', 'Regina SK', 'Calgary AB', 'Ottawa ON', 'Montreal QC']

==> Unique values in carrier:
['NorthStar Carriers', 'PrairieLine', 'QuebecExpress', 'MapleFreight Fleet', 'Contract Owner-Op']

==> Unique values in service_level:
['Standard', 'Express', 'Economy', 'standard', 'economy', 'express']

==> Unique values in goods_type:
['General Freight', 'Fragile', 'Refrigerated', 'Bulk', 'Hazardous']

==> Unique values in customer_priority_tier:
['Bronze', 'Silver', 'Gold']

==> Unique values in weather_condition:
['Fog', 'Snow', 'Rain', 'Clear', 'Storm', nan]


In [178]:
### checking statistics of numerical columns to spot suspicious values or unit errors
df.describe()

,shipment_month,distance_km,weight_kg,num_stops,is_cross_border,scheduled_transit_hours,pickup_delay_minutes,driver_experience_years,vehicle_age_years,traffic_index,route_congestion_score,prior_late_deliveries_30d,fuel_cost_cad,actual_transit_hours,delivered_late
count,6035.000000,6035.000000,6.035000e+03,6035.000000,6035.000000,6035.000000,6035.000000,5825.000000,6035.000000,5914.000000,6035.000000,6035.000000,5795.000000,6035.000000,6035.000000
mean,6.543662,632.192742,6.928251e+03,2.477051,0.113007,10.500388,20.367688,6.497648,5.489461,55.146365,0.449778,1.305551,180.696552,11.186038,0.205800
std,3.448260,319.246133,8.375515e+04,1.707457,0.316628,4.758576,30.618617,4.659277,3.433676,17.592978,0.194500,1.158943,58.759928,5.957114,0.404318
min,1.000000,80.500000,3.000000e+01,0.000000,0.000000,1.000000,-20.000000,0.200000,0.100000,5.000000,0.000000,0.000000,30.000000,0.880000,0.000000
25%,4.000000,398.600000,4.960000e+02,1.000000,0.000000,7.060000,-6.000000,3.200000,3.000000,43.300000,0.318000,0.000000,140.595000,7.080000,0.000000
50%,7.000000,571.100000,8.271000e+02,2.000000,0.000000,9.850000,18.000000,5.400000,4.800000,55.000000,0.446000,1.000000,181.180000,10.050000,0.000000
75%,10.000000,804.850000,1.286300e+03,4.000000,0.000000,13.175000,41.000000,8.600000,7.200000,67.200000,0.583000,2.000000,220.950000,14.010000,0.000000
max,12.000000,2556.700000,2.999300e+06,5.000000,1.000000,37.400000,146.000000,35.000000,20.000000,100.000000,1.000000,8.000000,392.010000,55.380000,1.000000


### Cleaning strategy
now I will apply data cleaning transformations based on my earlier exploratory findings. I will remove duplicate records, handle missing values with appropriate imputation methods, fix text spacing, and explicitly drop **actual_transit_hours** to prevent data leakage in my machine learning pipeline

In [179]:
### now I check initial shape before applying cleaning steps
print("==> Shape before cleaning:",df.shape)

### I drop fully duplicated rows from the dataset
df=df.drop_duplicates()

### next, I will drop duplicate shipment_id entries keeping the first record
### ==> I Googled parameters subset and keep <==
df=df.drop_duplicates(subset=['shipment_id'], keep='first')

### remove actual_transit_hours to prevent data leakage
df=df.drop(columns=['actual_transit_hours'])


### print shape after dropping duplicates and leakage column
print("==> Shape after removing duplicates and leakage column:",df.shape)

==> Shape before cleaning: (6035, 23)
==> Shape after removing duplicates and leakage column: (6000, 22)


In [180]:
### now I handle missing values in numerical columns using median imputation
num_cols=df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    # detection of null values
    if df[col].isnull().sum()>0:
        median_val=df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"==> '{col}' imputed with median: '{median_val}'")
### now I handle missing values in categorical columns using mode imputation
### ==> Googled selected_dtypes <==
cat_cols=df.select_dtypes(include=['str']).columns
for col in cat_cols:
    # every categorical feature except shipment_id
    if col!='shipment_id' and df[col].isnull().sum()>0:
        mode_val=df[col].mode()[0]
        df[col]=df[col].fillna(mode_val)
        print(f"==> '{col}' imputed with mode: '{mode_val}'")

==> 'driver_experience_years' imputed with median: '5.4'
==> 'traffic_index' imputed with median: '55.0'
==> 'fuel_cost_cad' imputed with median: '181.115'
==> 'weather_condition' imputed with mode: 'Clear'


In [183]:
### cleaning categorical string columns by stripping leading and trailing whitespaces
### ==> I Googled .strip() <==
string_cols=df.select_dtypes(include=['str']).columns
for col in string_cols:
    df[col]=df[col].astype(str).str.strip()

### checking if any missing values remain across all columns
remaining_nulls = df.isnull().sum().sum()
print("==> Remaining missing values:",remaining_nulls)

### display cleaned dataset
print('\n')
df.info()

==> Remaining missing values: 0


<class 'pandas.DataFrame'>
Index: 6000 entries, 0 to 6034
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   shipment_id                6000 non-null   str    
 1   origin_city                6000 non-null   str    
 2   destination_city           6000 non-null   str    
 3   carrier                    6000 non-null   str    
 4   service_level              6000 non-null   str    
 5   goods_type                 6000 non-null   str    
 6   customer_priority_tier     6000 non-null   str    
 7   shipment_month             6000 non-null   int64  
 8   distance_km                6000 non-null   float64
 9   weight_kg                  6000 non-null   float64
 10  num_stops                  6000 non-null   int64  
 11  is_cross_border            6000 non-null   int64  
 12  scheduled_transit_hours    6000 non-null   float64
 13  pickup_delay_minutes       600

### Insights
I have cleaned the dataset by removing duplicate rows, imputing missing numerical values with medians, filling missing categorical values with modes, and dropping **actual_transit_hours** to protect my model from data leakage. my dataset is now completely clean and ready for feature engineering